[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/laboratorios/S08/S08_02_random_forest.ipynb)

# S08 · 02 — Random Forest: la sabiduría de muchos árboles

**Tiempo estimado:** 55 min · **Sprint 8** · M3 (UD 3.7)

**Contexto TurisData.** El árbol podado nos dio un AUC de ≈ 0,72 y era muy inestable. Si en lugar de
**un** árbol preguntamos a **cientos** de árboles distintos y hacemos votar, los errores
individuales se compensan. Eso es un **bosque aleatorio** (*Random Forest*).

**Al terminar sabrás:**
1. Explicar el *bagging* (muestras con reemplazo) y la aleatoriedad en las variables.
2. Ajustar `n_estimators`, `max_depth` y `min_samples_leaf`, y usar la puntuación **OOB** como validación gratuita.
3. Comparar la importancia de impureza con la **importancia por permutación**.
4. Valorar el coste: rendimiento vs. explicabilidad y tiempo.

In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore", category=UserWarning)
SEMILLA = 42
sns.set_theme(style="whitegrid")

df = pd.read_csv(dato("reservas_turisdata.csv"), parse_dates=["fecha_reserva", "fecha_llegada"])
df["temporada_alta"] = df["fecha_llegada"].dt.month.isin([1, 2, 3, 7, 8, 12]).astype(int)
columnas_num = ["antelacion_dias", "noches", "adultos", "ninos", "precio_noche",
                "peticiones_especiales", "cliente_repetidor", "cancelaciones_previas", "temporada_alta"]
columnas_cat = ["tipo_habitacion", "regimen", "canal", "pais_origen", "tarifa", "deposito"]
X, y = df[columnas_num + columnas_cat], df["cancelada"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEMILLA, stratify=y)
cv = StratifiedKFold(5, shuffle=True, random_state=SEMILLA)

prep = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), columnas_num),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), columnas_cat),
])
Xtr = prep.fit_transform(X_train)
Xte = prep.transform(X_test)
nombres = prep.get_feature_names_out()

## 1. La idea: bagging + variables al azar

Un bosque construye muchos árboles y **promedia** sus predicciones. Para que los árboles sean
*distintos entre sí* (si fueran iguales, promediar no serviría), se introduce aleatoriedad de dos formas:
1. **Bagging** (*bootstrap aggregating*): cada árbol se entrena con una **muestra con reemplazo** de las reservas
   (unas aparecen repetidas y otras, ≈ 37 %, no aparecen).
2. En cada pregunta, cada árbol solo puede elegir entre un **subconjunto aleatorio de variables** (`max_features`).

Cada árbol individual es peor que uno bien podado, pero **sus errores no coinciden**; al votar, el ruido se
cancela y baja la varianza. Las reservas que un árbol no vio (**out-of-bag**, OOB) sirven para evaluar
sin necesidad de validación cruzada.

In [ ]:
# Cuántas reservas quedan fuera de una muestra bootstrap: ≈ 36,8 %
rng = np.random.default_rng(SEMILLA)
n = 5000
fuera = len(set(range(n)) - set(rng.integers(0, n, n))) / n
print(f"Reservas que NO entran en una muestra bootstrap: {fuera:.1%}")

**Ejercicio 1.** Entrena `bosque = RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
random_state=SEMILLA, n_jobs=2)` con train. Guarda `auc_test_bosque` (AUC en test con `predict_proba`) y
el AUC en test del árbol podado de la práctica anterior (`DecisionTreeClassifier(min_samples_leaf=80,
random_state=SEMILLA)`) en `auc_test_arbol`.

In [ ]:
# TODO: fit sobre Xtr, y_train; roc_auc_score(y_test, modelo.predict_proba(Xte)[:, 1]) para ambos
...
assert auc_test_bosque > auc_test_arbol, "El bosque debería superar a un único árbol podado"
assert auc_test_bosque > 0.70, "El AUC del bosque debería superar 0,70"
print(f"Árbol podado: {auc_test_arbol:.3f} · Bosque de 300 árboles: {auc_test_bosque:.3f}")

## 2. Cuántos árboles y de qué tamaño

- `n_estimators`: **más árboles casi nunca perjudica** (solo cuesta tiempo); se estabiliza.
- `min_samples_leaf` y `max_depth`: controlan el sobreajuste de cada árbol.
- `max_features`: nº de variables por pregunta (por defecto `sqrt` del total en clasificación).

Usamos la puntuación **OOB** (`oob_score=True`) para ver la evolución sin validación cruzada.

In [ ]:
n_arboles = [5, 10, 25, 50, 100, 200, 400]
oob_auc = []
for k in n_arboles:
    b = RandomForestClassifier(n_estimators=k, min_samples_leaf=5, oob_score=True,
                               random_state=SEMILLA, n_jobs=2).fit(Xtr, y_train)
    # oob_decision_function_ contiene la probabilidad OOB de cada reserva de train
    oob_auc.append(roc_auc_score(y_train, np.nan_to_num(b.oob_decision_function_[:, 1], nan=0.5)))
plt.figure(figsize=(7, 3.8))
plt.plot(n_arboles, oob_auc, marker="o", color="#2a6f97")
plt.xscale("log")
plt.title("Random Forest: AUC out-of-bag según el número de árboles")
plt.xlabel("Número de árboles (escala logarítmica)")
plt.ylabel("AUC out-of-bag")
plt.show()

**Ejercicio 2.** Con la lista `oob_auc`, guarda en `mejora_5_a_400` cuánto sube el AUC OOB al pasar de 5 a
400 árboles (`oob_auc[-1] - oob_auc[0]`) y en `mejora_100_a_400` lo que sube de 100 a 400 árboles.
Fíjate en que la segunda es mucho menor: los rendimientos son decrecientes.

In [ ]:
# TODO: los índices de n_arboles: 5 está en la posición 0, 100 en la 4 y 400 en la última
...
assert mejora_5_a_400 > 0.05, "De 5 a 400 árboles el AUC debería subir bastante"
assert abs(mejora_100_a_400) < mejora_5_a_400 / 2, "De 100 a 400 la mejora es mucho menor (se satura)"
print(f"5 → 400 árboles: +{mejora_5_a_400:.3f} · 100 → 400 árboles: {mejora_100_a_400:+.3f}")

## 3. Importancia de variables: dos formas de medirla

- **Importancia de impureza** (`feature_importances_`): rápida, pero sesgada hacia variables numéricas con
  muchos valores distintos (precio, antelación) y calculada **sobre train**.
- **Importancia por permutación**: se **baraja al azar** una columna del **test** y se mide cuánto
  empeora el AUC. Si empeora mucho, el modelo dependía de esa variable. Más fiable, pero más lenta.

Para poder leerla en columnas *originales* (no en cada one-hot), aplicamos la permutación a un
`Pipeline` completo.

In [ ]:
modelo_bosque = Pipeline([("prep", prep), ("rf", RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                                                        random_state=SEMILLA, n_jobs=2))])
modelo_bosque.fit(X_train, y_train)
perm = permutation_importance(modelo_bosque, X_test, y_test, scoring="roc_auc", n_repeats=5,
                              random_state=SEMILLA, n_jobs=1)
imp_perm = pd.Series(perm.importances_mean, index=X_test.columns).sort_values(ascending=False)
plt.figure(figsize=(7, 4.5))
imp_perm.iloc[::-1].plot(kind="barh", color="#2a6f97", xerr=pd.Series(perm.importances_std, index=X_test.columns)[imp_perm.iloc[::-1].index])
plt.title("Importancia por permutación (caída del AUC en test)")
plt.xlabel("Caída media del AUC al barajar la variable")
plt.ylabel("")
plt.tight_layout()
plt.show()

**Ejercicio 3.** Guarda en `variable_top` el nombre de la variable más importante según la permutación
(Series `imp_perm`, ya ordenada) y en `n_irrelevantes` cuántas variables tienen una importancia media
**menor de 0,002** (prácticamente nula).

In [ ]:
# TODO: imp_perm.index[0]; (imp_perm < 0.002).sum()
...
assert variable_top in X_test.columns, "Debe ser el nombre de una columna original"
assert variable_top == "antelacion_dias" or imp_perm.iloc[0] > 0.02, "La variable principal debería tener peso claro"
assert 1 <= n_irrelevantes <= len(imp_perm) - 3, "Debe haber algunas variables sin apenas aporte y otras con peso"
print(f"Variable más importante: {variable_top} · variables casi irrelevantes: {n_irrelevantes}")

## 4. Coste del bosque

El bosque gana en rendimiento pero **pierde explicabilidad** (300 árboles no se leen) y
cuesta más tiempo y memoria. Medimos ambas cosas frente al árbol.

In [ ]:
t0 = time.perf_counter()
RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=SEMILLA, n_jobs=2).fit(Xtr, y_train)
t_bosque = time.perf_counter() - t0
t0 = time.perf_counter()
DecisionTreeClassifier(min_samples_leaf=80, random_state=SEMILLA).fit(Xtr, y_train)
t_arbol = time.perf_counter() - t0
print(f"Tiempo de ajuste: árbol {t_arbol * 1000:.0f} ms · bosque {t_bosque * 1000:.0f} ms (≈ {t_bosque / t_arbol:.0f} veces más)")

## 5. Mini-reto: ¿compensa el bosque frente a la regresión logística?

Calcula con validación cruzada (5 folds, AUC, sobre **train**) el rendimiento de tres modelos y guárdalo
en `duelo` (DataFrame, filas `"arbol"`, `"bosque"` y `"logistica"`; columnas `auc_media` y `auc_desv`).
Para la logística usa un `Pipeline` con `StandardScaler` tras el mismo `prep` (o una versión con
escalado). Guarda en `mejor_modelo` el nombre del ganador y escribe en `comentario` (texto) una
frase sobre qué modelo elegirías teniendo en cuenta su explicabilidad.

In [ ]:
# TODO: tres pipelines con prep (la logística necesita StandardScaler después) y cross_val_score(..., scoring="roc_auc")
...
assert list(duelo.index) == ["arbol", "bosque", "logistica"], "Filas en el orden pedido"
assert duelo.loc["bosque", "auc_media"] > duelo.loc["arbol", "auc_media"], "El bosque debería ganar al árbol"
assert mejor_modelo in duelo.index
assert isinstance(comentario, str) and len(comentario) > 40
duelo

**Lección:** más complejo no siempre es mejor. En estos datos sintéticos la relación es prácticamente
«logística», así que la regresión logística es difícil de batir. Con relaciones no lineales o
interacciones complejas (datos reales), los bosques suelen ganar.

## 6. Para reflexionar

1. ¿Por qué promediar árboles **distintos** reduce el error, y por qué no serviría promediar árboles idénticos?
2. ¿Qué diferencia hay entre `feature_importances_` y la importancia por permutación? ¿Cuál usarías para explicar al cliente?
3. ¿Cuándo usarías la puntuación OOB en lugar de validación cruzada?
4. Si el bosque solo mejora 0,005 de AUC a la logística, pero es 50 veces más lento, ¿qué decides?

## 7. Resumen: qué has aprendido
- Bosque = muchos árboles con **bagging** y **variables al azar**, promediados.
- Más árboles no sobreajusta; se ajustan `min_samples_leaf`, `max_depth`, `max_features`.
- OOB da una validación «gratis»; la permutación da una importancia más fiable.
- Rendimiento, explicabilidad y coste van en tensión: hay que compararlos.